In [1]:
import torch

1. Verify PyTorch + GPU
    - Check torch.cuda.is_available().
    - Print the GPU name.
    - Goal: confirm tensors and model can run on cuda.

In [2]:
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no cuda')

2.13.0+cu126
True
NVIDIA GeForce GTX 1080 Ti



2. Define the tiny vocabulary
    - Use exactly three tokens: a, b, c.
    - Build stoi and itos.
    - Goal: decode(encode("abca")) == "abca".


In [3]:
stoi_dict = {"a": 0, "b": 1, "c": 2}
itos_dict = {0: "a", 1: "b", 2: "c"}

def stoi(s: str) -> int:
    return stoi_dict[s]

def itos(i: int) -> str:
    return itos_dict[i]

def encode(word: str) -> list[int]:
    return [stoi(s) for s in word]

def decode(nums: list[int]) -> str:
    return ''.join(itos(i) for i in nums)

vocab = ['a', 'b', 'c']
vocab_size = len(vocab)

In [4]:
enc = encode("abca")
dec = decode(enc)

print(enc, dec)

[0, 1, 2, 0] abca



3. Create a tiny training corpus
    - Start with a fixed string like: abacabaacbbccabacabaacbbcc
    - Keep this deterministic at first.
    - Goal: encoded data is a 1D tensor of token IDs.


In [5]:
text = "abacabaacbbccabacabaacbbcc"
data = torch.tensor(encode(text), dtype=torch.long)
data

tensor([0, 1, 0, 2, 0, 1, 0, 0, 2, 1, 1, 2, 2, 0, 1, 0, 2, 0, 1, 0, 0, 2, 1, 1,
        2, 2])

In [6]:
print(text)
print(encode(text))
print(data)
print(data.shape)
print(data.dtype)
print(data.min().item(), data.max().item())
print(decode(data.tolist()))

abacabaacbbccabacabaacbbcc
[0, 1, 0, 2, 0, 1, 0, 0, 2, 1, 1, 2, 2, 0, 1, 0, 2, 0, 1, 0, 0, 2, 1, 1, 2, 2]
tensor([0, 1, 0, 2, 0, 1, 0, 0, 2, 1, 1, 2, 2, 0, 1, 0, 2, 0, 1, 0, 0, 2, 1, 1,
        2, 2])
torch.Size([26])
torch.int64
0 2
abacabaacbbccabacabaacbbcc


In [7]:
decode(data.tolist()) == text

True


4. Build context-target examples
    - Choose block_size = 4.
    - Example:
        context: a b a c
        target:  a
    - Numerically:
        x = [0, 1, 0, 2]
        y = 0
    - Goal: produce batches shaped roughly:

    x: [batch_size, block_size]
    y: [batch_size]

In [8]:
block_size = 4
batch_size = 8 

def get_batch():
    ix = torch.randint(0, len(data) - block_size, (batch_size,))
    xb = torch.stack([data[i:i + block_size] for i in ix])
    yb = torch.stack([data[i + block_size] for i in ix])
    return xb, yb

xb, yb = get_batch()

print(xb.shape)  # should be [batch_size, block_size]
print(yb.shape)  # should be [batch_size]

for i in range(batch_size):
    context = decode(xb[i].tolist())
    target = decode([yb[i].item()])
    print(context, "->", target)

torch.Size([8, 4])
torch.Size([8])
cbbc -> c
baac -> b
caba -> c
ccab -> a
cbbc -> c
acab -> a
acbb -> c
caba -> a



5. Train a simple baseline first
    - Before GPT, build a tiny model:
        - token embedding
        - average or flatten context embeddings
        - linear layer to 3 logits
        - cross-entropy loss
    - Goal: understand logits, loss, backprop, optimizer, and
    generation.

In [9]:
import torch.nn as nn
import torch.nn.functional as F

n_embd = 8
device = "cuda" if torch.cuda.is_available() else "cpu"

class ContextModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.lm_head = nn.Linear(block_size * n_embd, vocab_size)

    def forward(self, idx, targets=None):
        x = self.token_embedding(idx)   # [B, T, C]
        B, T, C = x.shape
        x = x.reshape(B, T * C)         # [B, T*C]
        logits = self.lm_head(x)        # [B, vocab_size]

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits, targets)

        return logits, loss

In [10]:
model = ContextModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

for step in range(1000):
    xb, yb = get_batch()
    xb = xb.to(device)
    yb = yb.to(device)

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(step, loss.item())

0 1.1989128589630127
100 0.5058597326278687
200 0.13049376010894775
300 0.4610515832901001
400 0.03508079797029495
500 0.07845674455165863
600 0.07165887206792831
700 0.10106702148914337
800 0.007480279076844454
900 0.008559326641261578



6. Add generation
    - Start from a seed like "ab".
    - Predict next-token probabilities.
    - Sample or take argmax.
    - Append the predicted token.
    - Repeat.
    - Goal: generate strings using the trained model.

In [11]:
@torch.no_grad()
def generate(model, start, max_new_tokens):
    model.eval()

    idx = torch.tensor(encode(start), dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        idx_cond = idx[-block_size:]          # last block_size tokens
        idx_cond = idx_cond.unsqueeze(0)      # [T] -> [1, T]

        logits, loss = model(idx_cond)
        probs = F.softmax(logits, dim=-1)     # [1, vocab_size]

        idx_next = torch.multinomial(probs, num_samples=1)  # [1, 1]
        idx = torch.cat((idx, idx_next.squeeze(0)), dim=0)

    model.train()
    return decode(idx.tolist())

In [12]:
print(generate(model, "abac", 20))

abacabaacbbccabacabacaba



7. Upgrade to mini GPT
    - Add token embeddings.
    - Add positional embeddings.
    - Add one causal self-attention block.
    - Add feed-forward layer.
    - Add residual connections and layer norm.
    - Add final linear head to predict 3 logits.
    - Suggested tiny settings:

    vocab_size = 3
    block_size = 8
    n_embd = 16
    n_head = 2
    n_layer = 1

In [22]:
def get_batch():
    ix = torch.randint(0, len(data) - block_size, (batch_size,))

    xb = torch.stack([data[i:i + block_size] for i in ix])
    yb = torch.stack([data[i + 1:i + block_size + 1] for i in ix])

    return xb, yb

get_batch()

(tensor([[2, 1, 1, 2, 2, 0, 1, 0],
         [0, 2, 1, 1, 2, 2, 0, 1],
         [2, 0, 1, 0, 2, 0, 1, 0],
         [0, 1, 0, 2, 0, 1, 0, 0],
         [0, 2, 0, 1, 0, 0, 2, 1],
         [2, 0, 1, 0, 2, 0, 1, 0],
         [2, 0, 1, 0, 0, 2, 1, 1],
         [0, 1, 0, 0, 2, 1, 1, 2]]),
 tensor([[1, 1, 2, 2, 0, 1, 0, 2],
         [2, 1, 1, 2, 2, 0, 1, 0],
         [0, 1, 0, 2, 0, 1, 0, 0],
         [1, 0, 2, 0, 1, 0, 0, 2],
         [2, 0, 1, 0, 0, 2, 1, 1],
         [0, 1, 0, 2, 0, 1, 0, 0],
         [0, 1, 0, 0, 2, 1, 1, 2],
         [1, 0, 0, 2, 1, 1, 2, 2]]))

In [25]:
block_size = 8
batch_size = 8
n_embd = 16
n_head = 2
n_layer = 1

class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding(idx)              # [B, T, C]
        pos = torch.arange(T, device=idx.device)         # [T]
        pos_emb = self.position_embedding(pos)           # [T, C]

        x = tok_emb + pos_emb                            # [B, T, C]
        logits = self.lm_head(x)                         # [B, T, vocab_size]

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(
                logits.view(B * T, C),
                targets.view(B * T),
            )

        return logits, loss


In [26]:
model = MiniGPT().to(device)

xb, yb = get_batch()
xb = xb.to(device)
yb = yb.to(device)

logits, loss = model(xb, yb)

print(xb.shape)      # [8, 8]
print(yb.shape)      # [8, 8]
print(logits.shape)  # [8, 8, 3]
print(loss.item())

torch.Size([8, 8])
torch.Size([8, 8])
torch.Size([8, 8, 3])
1.3310673236846924



8. Compare models
    - Compare:
    - frequency baseline
    - embedding + linear model
    - mini GPT
    - Goal: see whether the transformer adds anything on this tiny
    dataset.

9. Overfit intentionally
    - Use a very small corpus.
    - Train until loss becomes very low.
    - Goal: prove the model can memorize before asking it to
    generalize.

10. Then vary one thing at a time

- Increase block_size.
- Increase corpus length.
- Add randomness to data.
- Change sampling temperature.
- Goal: understand cause and effect.